# 4 智能体评估与调试

In [1]:
import torch
assert torch.cuda.is_available()

from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = r"F:\agent\Qwen2.5-0.5B"

# 加载分词器和模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on {device}")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 3880.76it/s]


Model loaded on cuda


In [3]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total model parameters: {total_params / 1e6 :.2f}M")
print(model)

Total model parameters: 494.03M
Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), ep

In [4]:
from transformers import pipeline

text_generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text=False,
    device = device
)

prompt = \
"""
怪奇物语是谁写的
""" 
print(prompt)

generated_text = text_generator(
    prompt,
    max_length=2000,
    min_length=50,
    do_sample=True,
    early_stopping=True
)[0]['generated_text']

print("生成的文本：")
print(generated_text)

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'early_stopping', 'min_length', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.



怪奇物语是谁写的



[transformers] Both `max_new_tokens` (=256) and `max_length`(=2000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


生成的文本：
《怪奇物语》（英语：Strange Adventures）是一部以现代日本的都市为背景，讲述男女主角在异国他乡相遇、相识、相爱的故事。这部作品由日本漫画家野田宽创作，并于1985年首次出版。故事主要围绕着男女主角之间的爱情和友情展开。

《怪奇物语》在日本获得了巨大的成功，它不仅在国内市场取得了巨大成功，还在国际上也获得了广泛的认可和好评。这部作品在全球范围内被翻译成多种语言，在海外产生了深远的影响。

尽管《怪奇物语》是日本漫画家野田宽的作品，但它在全世界范围内广受喜爱，成为了日本动漫文化的重要组成部分之一。它的影响力不仅仅局限于日本，而且已经传播到了全球许多国家和地区。


## 人工测试

In [6]:
from transformers import pipeline

text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer = tokenizer,
    return_full_text = False,
    device = device,
)
prompt = \
"""
from typing import List
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    # Check if in given list of numbers, are any two numbers closer to each other than given threshold.
    # >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    # False
    # >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    # True
    # Your Code Here
""" 
generated_text = text_generator(
    prompt,
    max_length = 2000,
    min_length = 50,
    do_sample = True,
    early_stopping = True,
)[0]["generated_text"]

print(generated_text)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


    for i in range(len(numbers)):
        for j in range(i+1, len(numbers)):
            if abs(numbers[i] - numbers[j]) <= threshold:
                return True
    return False

# Test Cases
print(has_close_elements([1.0, 2.0, 3.0], 0.5))  # Expected output: False
print(has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3))  # Expected output: True
```

In this solution, the `has_close_elements` function takes a list of floating-point numbers and a threshold value as input. It iterates through all possible pairs of elements in the list using nested loops and checks if their difference is less than or equal to the threshold. If such a pair is found, it returns `True`. Otherwise, after checking all pairs, it returns `False`. The test cases demonstrate how the function works with different inputs. 

The time complexity of this approach is O(n^2), where n is the number of elements in the list. This is because there are at most n*(n-1)/2 comparisons between each element


## 基准数据集测试

### HumanEval

In [7]:
# 定义函数用于模型input and output
def generate_one_completion(prompt):
    generated_text = text_generator(
        prompt,
        max_length=2000,
        min_length = 50,
        return_full_text = False,
        do_sample=True,
        early_stopping = True
    )[0]["generated_text"]

    return generated_text

In [10]:
# 加载humaneval的数据集进入
from human_eval.data import write_jsonl, read_problems
from tqdm import tqdm
print("Loading prombles...")
problems = read_problems()
print(f"Loaded {len(problems)} problems.")

Loading prombles...
Loaded 164 problems.


In [11]:
num_samples_per_task = 1
samples = []
keys = list(problems.keys())
num = len(keys)
for i in tqdm(range(num)):
    completion = generate_one_completion(problems[keys[i]]["prompt"])
    samples.append(dict(task_id = keys[i], completion = completion))
    

  5%|▌         | 9/164 [00:27<05:54,  2.28s/it][transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 164/164 [19:56<00:00,  7.30s/it]


In [12]:
print("Writing samples to JSONL file...")
write_jsonl("samples.jsonl", samples)

Writing samples to JSONL file...


In [17]:
import os

os.system('powershell -command "Get-Content samples.jsonl -First 10"')

0

In [1]:
from human_eval.data import read_problems, write_jsonl
from human_eval.evaluation import evaluate_functional_correctness

results = evaluate_functional_correctness(
    sample_file="samples.jsonl",
    k=[1],  # 计算 pass@1
    n_workers=1,
    timeout=10.0
)

print(f"Pass@1: {results['pass@1']:.2%}")

Reading samples...


164it [00:00, 38631.13it/s]


Running test suites...


100%|██████████| 164/164 [00:38<00:00,  4.22it/s]


Writing results to samples.jsonl_results.jsonl...


100%|██████████| 164/164 [00:00<00:00, 8180.60it/s]

Pass@1: 0.00%


In [ ]:
import os
lm_eval_path = "./lm_eval"
model_path = r"F:\agent\Qwen2.5-0.5B"
os.system("cd lm-evaluation-harness")
# 评估模型在 MMLU 上的性能
os.system(f"lm_eval --model hf     --model_args pretrained={model_path}     --tasks mmlu     --device cuda:0     --batch_size 8")

### 因为Windows使用os.system方法输出不了日志，故使用命令行启动将结果粘贴如下
2026-08-15:17:44:51 INFO     [loggers.evaluation_tracker:316] Output path not provided, skipping saving results aggregated
hf ({'pretrained': 'F:\\agent\\Qwen2.5-0.5B'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |   |0.4571|±  |0.0041|
| - humanities                          |      2|none  |     0|acc   |↑  |0.4225|±  |0.0070|
|  - formal_logic                       |      1|none  |     0|acc   |↑  |0.3333|±  |0.0422|
|  - high_school_european_history       |      1|none  |     0|acc   |↑  |0.6000|±  |0.0383|
|  - high_school_us_history             |      1|none  |     0|acc   |↑  |0.5392|±  |0.0350|
|  - high_school_world_history          |      1|none  |     0|acc   |↑  |0.6034|±  |0.0318|
|  - international_law                  |      1|none  |     0|acc   |↑  |0.7190|±  |0.0410|
|  - jurisprudence                      |      1|none  |     0|acc   |↑  |0.5833|±  |0.0477|
|  - logical_fallacies                  |      1|none  |     0|acc   |↑  |0.4724|±  |0.0392|
|  - moral_disputes                     |      1|none  |     0|acc   |↑  |0.5231|±  |0.0269|
|  - moral_scenarios                    |      1|none  |     0|acc   |↑  |0.2380|±  |0.0142|
|  - philosophy                         |      1|none  |     0|acc   |↑  |0.4759|±  |0.0284|
|  - prehistory                         |      1|none  |     0|acc   |↑  |0.5463|±  |0.0277|
|  - professional_law                   |      1|none  |     0|acc   |↑  |0.3572|±  |0.0122|
|  - world_religions                    |      1|none  |     0|acc   |↑  |0.5848|±  |0.0378|
| - other                               |      2|none  |     0|acc   |↑  |0.5063|±  |0.0088|
|  - business_ethics                    |      1|none  |     0|acc   |↑  |0.5300|±  |0.0502|
|  - clinical_knowledge                 |      1|none  |     0|acc   |↑  |0.5019|±  |0.0308|
|  - college_medicine                   |      1|none  |     0|acc   |↑  |0.4393|±  |0.0378|
|  - global_facts                       |      1|none  |     0|acc   |↑  |0.2800|±  |0.0451|
|  - human_aging                        |      1|none  |     0|acc   |↑  |0.5426|±  |0.0334|
|  - management                         |      1|none  |     0|acc   |↑  |0.5728|±  |0.0490|
|  - marketing                          |      1|none  |     0|acc   |↑  |0.7436|±  |0.0286|
|  - medical_genetics                   |      1|none  |     0|acc   |↑  |0.5100|±  |0.0502|
|  - miscellaneous                      |      1|none  |     0|acc   |↑  |0.5556|±  |0.0178|
|  - nutrition                          |      1|none  |     0|acc   |↑  |0.5850|±  |0.0282|
|  - professional_accounting            |      1|none  |     0|acc   |↑  |0.3298|±  |0.0280|
|  - professional_medicine              |      1|none  |     0|acc   |↑  |0.3676|±  |0.0293|
|  - virology                           |      1|none  |     0|acc   |↑  |0.4277|±  |0.0385|
| - social sciences                     |      2|none  |     0|acc   |↑  |0.5291|±  |0.0089|
|  - econometrics                       |      1|none  |     0|acc   |↑  |0.3158|±  |0.0437|
|  - high_school_geography              |      1|none  |     0|acc   |↑  |0.5556|±  |0.0354|
|  - high_school_government_and_politics|      1|none  |     0|acc   |↑  |0.5337|±  |0.0360|
|  - high_school_macroeconomics         |      1|none  |     0|acc   |↑  |0.4359|±  |0.0251|
|  - high_school_microeconomics         |      1|none  |     0|acc   |↑  |0.4790|±  |0.0324|
|  - high_school_psychology             |      1|none  |     0|acc   |↑  |0.6257|±  |0.0207|
|  - human_sexuality                    |      1|none  |     0|acc   |↑  |0.5420|±  |0.0437|
|  - professional_psychology            |      1|none  |     0|acc   |↑  |0.4592|±  |0.0202|
|  - public_relations                   |      1|none  |     0|acc   |↑  |0.5273|±  |0.0478|
|  - security_studies                   |      1|none  |     0|acc   |↑  |0.5592|±  |0.0318|
|  - sociology                          |      1|none  |     0|acc   |↑  |0.6716|±  |0.0332|
|  - us_foreign_policy                  |      1|none  |     0|acc   |↑  |0.7200|±  |0.0451|
| - stem                                |      2|none  |     0|acc   |↑  |0.3901|±  |0.0085|
|  - abstract_algebra                   |      1|none  |     0|acc   |↑  |0.3300|±  |0.0473|
|  - anatomy                            |      1|none  |     0|acc   |↑  |0.4222|±  |0.0427|
|  - astronomy                          |      1|none  |     0|acc   |↑  |0.4671|±  |0.0406|
|  - college_biology                    |      1|none  |     0|acc   |↑  |0.4375|±  |0.0415|
|  - college_chemistry                  |      1|none  |     0|acc   |↑  |0.2900|±  |0.0456|
|  - college_computer_science           |      1|none  |     0|acc   |↑  |0.3500|±  |0.0479|
|  - college_mathematics                |      1|none  |     0|acc   |↑  |0.2900|±  |0.0456|
|  - college_physics                    |      1|none  |     0|acc   |↑  |0.3137|±  |0.0462|
|  - computer_security                  |      1|none  |     0|acc   |↑  |0.6900|±  |0.0465|
|  - conceptual_physics                 |      1|none  |     0|acc   |↑  |0.3872|±  |0.0318|
|  - electrical_engineering             |      1|none  |     0|acc   |↑  |0.5103|±  |0.0417|
|  - elementary_mathematics             |      1|none  |     0|acc   |↑  |0.3254|±  |0.0241|
|  - high_school_biology                |      1|none  |     0|acc   |↑  |0.5387|±  |0.0284|
|  - high_school_chemistry              |      1|none  |     0|acc   |↑  |0.3892|±  |0.0343|
|  - high_school_computer_science       |      1|none  |     0|acc   |↑  |0.4400|±  |0.0499|
|  - high_school_mathematics            |      1|none  |     0|acc   |↑  |0.3111|±  |0.0282|
|  - high_school_physics                |      1|none  |     0|acc   |↑  |0.2450|±  |0.0351|
|  - high_school_statistics             |      1|none  |     0|acc   |↑  |0.3102|±  |0.0315|
|  - machine_learning                   |      1|none  |     0|acc   |↑  |0.4107|±  |0.0467|

|      Groups      |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu              |      2|none  |      |acc   |   |0.4571|±  |0.0041|
| - humanities     |      2|none  |     0|acc   |↑  |0.4225|±  |0.0070|
| - other          |      2|none  |     0|acc   |↑  |0.5063|±  |0.0088|
| - social sciences|      2|none  |     0|acc   |↑  |0.5291|±  |0.0089|
| - stem           |      2|none  |     0|acc   |↑  |0.3901|±  |0.0085|

In [ ]:
import os
lm_eval_path = "lm_eval"
model_path = "../models/Qwen2.5-0.5B-Instruct/"
os.system("cd lm-evaluation-harness")
# 评估模型在 GSM8K 上的性能
os.system(f"lm_eval --model hf     --model_args pretrained={model_path}     --tasks gsm8k     --device cuda:0     --batch_size 8")

hf ({'pretrained': 'F:\\agent\\Qwen2.5-0.5B'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 8
|Tasks|Version|     Filter     |n-shot|  Metric   |   |Value |   |Stderr|
|-----|------:|----------------|-----:|-----------|---|-----:|---|-----:|
|gsm8k|      3|flexible-extract|     5|exact_match|↑  |0.3199|±  |0.0128|
|     |       |strict-match    |     5|exact_match|↑  |0.2199|±  |0.0114|

In [ ]:
import os
lm_eval_path = "lm_eval"
model_path = "../models/Qwen2.5-0.5B-Instruct/"
os.environ["HF_ALLOW_CODE_EVAL"] = "1"
os.system("cd lm-evaluation-harness")
os.system(f"lm_eval --model hf     --model_args pretrained={model_path}     --tasks humaneval     --device cuda:0     --batch_size 8 --confirm_run_unsafe_code")

code_eval 指标不支持 Windows

## LLM自动评测

In [3]:
llm_as_a_judge_prompt_pre = \
"""
Prompt:

You are a code review expert responsible for evaluating the quality of the following code. Please assess the generated code based on the following criteria and provide a score from 1 to 10, where 1 means the code quality is very poor, and 10 means the code quality is excellent.

Evaluation Criteria:

Correctness: Does the code correctly solve the problem and is it free of syntax or runtime errors?

Conciseness: Is the code concise and efficient, avoiding unnecessary implementations?

Readability: Is the code clear and understandable, with descriptive variable names and sufficient comments?

Performance: Does the code consider performance optimization and avoid potential performance bottlenecks?

Scalability: Is the code easily extensible for future modifications or feature additions?

Code:
"""

llm_as_a_judge_prompt_post = \
"""

Scoring Guidelines:

1-3: The code has significant errors, does not work properly, or is overly complex and difficult to understand.

4-6: The code solves the problem but has some redundancy or areas for improvement in terms of performance, readability, or structure.

7-9: The code is correct and efficient, well-structured, and easy to understand, with minor room for improvement.

10: The code is perfect, adhering to best practices, concise, efficient, readable, and performs well.

Evaluation Result:

Score:

Review Explanation:
"""

In [ ]:
from transformers import pipeline
text_generator = pipeline(
    "text-generation",
    model = model,
    tokenizer = tokenizer,
    return_full_text = False,
    device = device
)
prompt = \
"""
```python
from typing import List
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    # Check if in given list of numbers, are any two numbers closer to each other than given threshold.
    # >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    # False
    # >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    # True
    for i in range(len(numbers)):
        for j in range(i+1, len(numbers)):
            if abs(numbers[i] - numbers[j]) < threshold:
                return True
    return False
```
This solution iterates through all possible pairs of numbers in the given list, calculates their absolute difference using the `abs` function, and checks if this difference is less than the provided threshold. If such a pair exists, it returns True; otherwise, it returns False after checking all pairs. The check function with example usage demonstrates its correctness by verifying the presence of close elements in a sample list. ```
"""

prompt = llm_as_a_judge_prompt_pre + prompt + llm_as_a_judge_prompt_post

# 生成续写文本
generated_text = text_generator(
    prompt,
    max_length=2000, 
    min_length=50,  
    do_sample=True,  
    early_stopping=True  
)[0]['generated_text']

print("生成的文本：")
print(generated_text)

[transformers] The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'min_length', 'do_sample', 'max_length', 'early_stopping'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=2000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=Fa

生成的文本：
The code snippet you've provided appears to be intended to find pairs of numbers in a list that are closest together (i.e., have the same absolute difference). However, there are several issues with this implementation:

1. **Incorrect Function Name**: The function name `has_close_elements` suggests it's meant to handle arrays of floating-point numbers rather than lists of integers. It also lacks documentation, which is crucial for others reading the code.

2. **Logic Flaw**: The logic within the loop (`for i in range(len(numbers)):` and `for j in range(i+1, len(numbers)):`) is flawed because it incorrectly compares each element against every other element, leading to incorrect results.

3. **Efficiency**: Even though the function works as intended, it might still be slow due to the nested loops, especially when dealing with large datasets.

4. **Readability**: The comments are minimal and unclear, making it hard to understand what the function is supposed to do.

5. **Performan

## 调试

In [5]:
from transformers import pipeline
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer = tokenizer,
    return_full_text = False,
    device = device
)

prompt = """
from typing import List
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    # Check if in given list of numbers, are any two numbers closer to each other than given threshold.
    # >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    # False
    # >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    # True
    # Your Code Here
"""

generated_text = text_generator(
    prompt,
    max_length = 2000,
    min_length = 50,
    do_sample = True,
    early_stopping = True
)[0]['generated_text']

print("生成的文本：")
print(generated_text)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


生成的文本：
    for i in range(len(numbers)-1):
        for j in range(i+1, len(numbers)):
            if abs(numbers[i] - numbers[j]) <= threshold:
                return True
    return False

# Test cases
print(has_close_elements([1.0, 2.0, 3.0], 0.5))  # Expected output: False
print(has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3))  # Expected output: True
```

This solution uses a nested loop to compare every pair of elements in the list with every other element except itself and its next one. If any such pair satisfies the condition (i.e., their absolute difference is less than or equal to the given threshold), it returns `True`. Otherwise, after checking all pairs, it returns `False` if no such pair was found.

Please note that this code assumes that the input list contains at least two elements. If there's only one element, the function will return `False` as per the problem statement. The test cases provided cover both scenarios. Feel free to adjust them according to your ne

In [7]:
from transformers import pipeline

# 创建文本生成管道
text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    device=device  # 指定设备
)
prompt = """
Generate a Python function that checks if there are any two numbers in a given list that are closer to each other than a specified threshold. The function should take a list of floating-point numbers and a threshold value as input, and return True if such pairs exist, otherwise return False. Include example usage in the docstring.
from typing import List
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    # Check if in given list of numbers, are any two numbers closer to each other than given threshold.
    # >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    # False
    # >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    # True
    # Your Code Here
    # You should only reponse the code
"""
generated_text = text_generator(
    prompt,
    max_length = 2000,
    min_length = 50,
    do_sample = True,
    early_stopping = True
)[0]['generated_text']
print("生成的文本：")
print(generated_text)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=2000) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


生成的文本：
    pass

# Test cases to verify the correctness of your function
assert not has_close_elements([1.0, 2.0, 3.0], 0.5), "Test Case 1 Failed"
assert has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3), "Test Case 2 Failed"
print("All test cases passed!")

```
